In [0]:
# NBA SHOT IQ - BRONZE
# Preparacion de la arquitectura Medallion

catalogo = spark.sql("SELECT current_catalog()").first()[0]

spark.sql("CREATE SCHEMA IF NOT EXISTS nba_shot_iq")

print("Catalogo activo:", catalogo)
print("Esquema creado: nba_shot_iq")
print("Listo para cargar los datos originales")

Catalogo activo: workspace
Esquema creado: nba_shot_iq
Listo para cargar los datos originales


In [0]:
import requests

url = "https://api.github.com/repos/sportsdataverse/sportsdataverse-data/releases/tags/nba_stats_shots"

respuesta = requests.get(url, timeout=30)
respuesta.raise_for_status()

archivos = respuesta.json()["assets"]

for anio in [2024, 2025, 2026]:
    nombre = f"shots_{anio}.parquet"
    encontrados = [
        archivo for archivo in archivos
        if archivo["name"] == nombre
    ]

    if encontrados:
        archivo = encontrados[0]
        print(anio, nombre, archivo["size"], "bytes")
        print(archivo["browser_download_url"])
    else:
        print("NO ENCONTRADO:", nombre)

2024 shots_2024.parquet 4271538 bytes
https://github.com/sportsdataverse/sportsdataverse-data/releases/download/nba_stats_shots/shots_2024.parquet
2025 shots_2025.parquet 4297616 bytes
https://github.com/sportsdataverse/sportsdataverse-data/releases/download/nba_stats_shots/shots_2025.parquet
2026 shots_2026.parquet 4261369 bytes
https://github.com/sportsdataverse/sportsdataverse-data/releases/download/nba_stats_shots/shots_2026.parquet


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.nba_shot_iq")

spark.sql("SHOW TABLES IN workspace.nba_shot_iq").show()

+-----------+---------------+-----------+
|   database|      tableName|isTemporary|
+-----------+---------------+-----------+
|nba_shot_iq|   bronze_shots|      false|
|nba_shot_iq| gold_jugadores|      false|
|nba_shot_iq|gold_temporadas|      false|
|nba_shot_iq|     gold_zonas|      false|
|nba_shot_iq|   silver_shots|      false|
+-----------+---------------+-----------+



In [0]:
import requests

base = (
    "https://github.com/sportsdataverse/"
    "sportsdataverse-data/releases/download/nba_stats_shots/"
)

for anio in [2024, 2025, 2026]:
    url = base + f"shots_{anio}.parquet"
    respuesta = requests.get(
        url,
        headers={"Range": "bytes=0-3"},
        stream=True,
        timeout=30
    )
    print(f"shots_{anio}.parquet: HTTP {respuesta.status_code}")
    respuesta.close()

shots_2024.parquet: HTTP 206
shots_2025.parquet: HTTP 206
shots_2026.parquet: HTTP 206


In [0]:
import requests
import os

base = (
    "https://github.com/sportsdataverse/"
    "sportsdataverse-data/releases/download/nba_stats_shots/"
)

carpeta = "/Volumes/workspace/nba_shot_iq/raw_nba_shots"

for anio in [2024, 2025, 2026]:
    nombre = f"shots_{anio}.parquet"
    destino = f"{carpeta}/{nombre}"

    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        print(f"Ya existe: {nombre}")
        continue

    print(f"Descargando {nombre}...")

    with requests.get(base + nombre, stream=True, timeout=120) as r:
        r.raise_for_status()

        with open(destino, "wb") as archivo:
            for bloque in r.iter_content(chunk_size=1024 * 1024):
                if bloque:
                    archivo.write(bloque)

    print(f"Guardado: {nombre} ({os.path.getsize(destino):,} bytes)")

print("Descargas terminadas.")

Ya existe: shots_2024.parquet
Ya existe: shots_2025.parquet
Ya existe: shots_2026.parquet
Descargas terminadas.


In [0]:
from pyspark.sql import functions as F

ruta = "/Volumes/workspace/nba_shot_iq/raw_nba_shots"

df_bronze = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(
        f"{ruta}/shots_2024.parquet",
        f"{ruta}/shots_2025.parquet",
        f"{ruta}/shots_2026.parquet"
    )
)

print("Total de registros:", df_bronze.count())
print("Total de columnas:", len(df_bronze.columns))

df_bronze.printSchema()

display(df_bronze.limit(10))

Total de registros: 703805
Total de columnas: 19
root
 |-- game_id: string (nullable = true)
 |-- season: integer (nullable = true)
 |-- season_type_id: string (nullable = true)
 |-- period: long (nullable = true)
 |-- clock: string (nullable = true)
 |-- team_id: long (nullable = true)
 |-- team_tricode: string (nullable = true)
 |-- person_id: long (nullable = true)
 |-- player_name: string (nullable = true)
 |-- action_type: string (nullable = true)
 |-- sub_type: string (nullable = true)
 |-- shot_result: string (nullable = true)
 |-- shot_value: long (nullable = true)
 |-- shot_distance: long (nullable = true)
 |-- x_legacy: long (nullable = true)
 |-- y_legacy: long (nullable = true)
 |-- description: string (nullable = true)
 |-- score_home: string (nullable = true)
 |-- score_away: string (nullable = true)



game_id,season,season_type_id,period,clock,team_id,team_tricode,person_id,player_name,action_type,sub_type,shot_result,shot_value,shot_distance,x_legacy,y_legacy,description,score_home,score_away
0022400001,2025,2,1,PT11M43.00S,1610612737,ATL,1642258,Risacher,Missed Shot,Jump Shot,Missed,3,27,-168,205,MISS Risacher 27' 3PT Jump Shot,,
0022400001,2025,2,1,PT11M38.00S,1610612737,ATL,1630552,Johnson,Missed Shot,Driving Floating Bank Jump Shot,Missed,2,14,-136,-1,MISS Johnson 14' Driving Floating Bank Jump Shot,,
0022400001,2025,2,1,PT11M24.00S,1610612738,BOS,1628401,White,Missed Shot,Jump Shot,Missed,3,28,-181,218,MISS White 28' 3PT Jump Shot,,
0022400001,2025,2,1,PT10M50.00S,1610612737,ATL,1630552,Johnson,Made Shot,Jump Shot,Made,3,26,157,203,Johnson 26' 3PT Jump Shot (3 PTS) (Wallace 1 AST),0,3
0022400001,2025,2,1,PT10M35.00S,1610612738,BOS,1627759,Brown,Made Shot,Pullup Jump shot,Made,3,27,102,253,Brown 27' 3PT Pullup Jump Shot (3 PTS) (White 1 AST),3,3
0022400001,2025,2,1,PT10M06.00S,1610612738,BOS,1627759,Brown,Missed Shot,Jump Shot,Missed,3,24,-228,85,MISS Brown 24' 3PT Jump Shot,,
0022400001,2025,2,1,PT09M47.00S,1610612737,ATL,1630811,Wallace,Missed Shot,Jump Shot,Missed,3,25,-176,184,MISS Wallace 25' 3PT Jump Shot,,
0022400001,2025,2,1,PT09M44.00S,1610612737,ATL,203991,Capela,Made Shot,Putback Layup Shot,Made,2,3,-25,8,Capela 3' Putback Layup (2 PTS),3,5
0022400001,2025,2,1,PT09M35.00S,1610612738,BOS,1627759,Brown,Made Shot,Driving Layup Shot,Made,2,1,5,13,Brown 1' Driving Layup (5 PTS) (White 2 AST),5,5
0022400001,2025,2,1,PT09M23.00S,1610612737,ATL,1642258,Risacher,Made Shot,Cutting Layup Shot,Made,2,2,10,13,Risacher 2' Cutting Layup Shot (2 PTS) (Wallace 2 AST),5,7


In [0]:
# Guardar los datos originales en la capa Bronze
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.nba_shot_iq.bronze_shots")
)

# Comprobar que la tabla se creó correctamente
df_verificacion = spark.table(
    "workspace.nba_shot_iq.bronze_shots"
)

print("BRONZE CREADA CORRECTAMENTE")
print("Registros:", df_verificacion.count())
print("Columnas:", len(df_verificacion.columns))

display(
    df_verificacion.groupBy("season")
    .count()
    .orderBy("season")
)

BRONZE CREADA CORRECTAMENTE
Registros: 703805
Columnas: 19


season,count
2024,233789
2025,235140
2026,234876


In [0]:
from pyspark.sql import functions as F

# Leer la tabla Bronze original
bronze = spark.table("workspace.nba_shot_iq.bronze_shots")

# Eliminar duplicados exactos
silver = bronze.dropDuplicates(bronze.columns)

# Normalizar textos de categorías
for columna in [
    "team_tricode",
    "player_name",
    "action_type",
    "sub_type",
    "shot_result"
]:
    silver = silver.withColumn(
        columna,
        F.trim(F.col(columna))
    )

# Convertir marcadores vacíos a nulos
for columna in ["score_home", "score_away"]:
    silver = silver.withColumn(
        columna,
        F.when(
            F.trim(F.col(columna)) == "",
            F.lit(None)
        ).otherwise(F.trim(F.col(columna)))
    )

# Crear variables para el análisis
silver = (
    silver
    .withColumn(
        "is_made",
        F.when(F.lower(F.col("shot_result")) == "made", 1)
         .when(F.lower(F.col("shot_result")) == "missed", 0)
    )
    .withColumn(
        "shot_points",
        F.when(F.col("is_made") == 1, F.col("shot_value"))
         .otherwise(F.lit(0))
    )
    .withColumn(
        "shot_zone",
        F.when(F.col("shot_distance") < 5, "Cerca del aro")
         .when(F.col("shot_distance") < 15, "Distancia media")
         .otherwise("Distancia larga")
    )
)

# Guardar la tabla Silver en Delta
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.nba_shot_iq.silver_shots")
)

# Verificar resultados
total_bronze = bronze.count()
total_silver = spark.table(
    "workspace.nba_shot_iq.silver_shots"
).count()

print("BRONZE:", total_bronze)
print("SILVER:", total_silver)
print("DUPLICADOS ELIMINADOS:", total_bronze - total_silver)

BRONZE: 703805
SILVER: 703781
DUPLICADOS ELIMINADOS: 24


In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.nba_shot_iq.silver_shots")

# 1. Eficiencia por temporada
gold_temporadas = (
    silver.groupBy("season")
    .agg(
        F.count("*").alias("intentos"),
        F.sum("is_made").alias("encestados"),
        F.round(F.avg("is_made") * 100, 2).alias("efectividad_pct"),
        F.sum("shot_points").alias("puntos_generados"),
        F.round(F.avg("shot_distance"), 2).alias("distancia_promedio")
    )
)

# 2. Eficiencia por zona
gold_zonas = (
    silver.groupBy("season", "shot_zone")
    .agg(
        F.count("*").alias("intentos"),
        F.sum("is_made").alias("encestados"),
        F.round(F.avg("is_made") * 100, 2).alias("efectividad_pct")
    )
)

# 3. Eficiencia por jugador y temporada
gold_jugadores = (
    silver.groupBy("season", "person_id", "player_name")
    .agg(
        F.count("*").alias("intentos"),
        F.sum("is_made").alias("encestados"),
        F.round(F.avg("is_made") * 100, 2).alias("efectividad_pct"),
        F.sum("shot_points").alias("puntos_generados")
    )
)

# Guardar las tres tablas Gold en Delta
for nombre, df in [
    ("gold_temporadas", gold_temporadas),
    ("gold_zonas", gold_zonas),
    ("gold_jugadores", gold_jugadores)
]:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.nba_shot_iq.{nombre}")
    )
    print("Tabla creada:", nombre)

# Mostrar resultados
display(
    spark.table("workspace.nba_shot_iq.gold_temporadas")
    .orderBy("season")
)

Tabla creada: gold_temporadas
Tabla creada: gold_zonas
Tabla creada: gold_jugadores


season,intentos,encestados,efectividad_pct,puntos_generados,distancia_promedio
2024,233776,110696,47.35,255049,14.04
2025,235132,109665,46.64,254895,14.51
2026,234873,110338,46.98,255614,14.3


In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.nba_shot_iq.bronze_shots")
silver = spark.table("workspace.nba_shot_iq.silver_shots")
gold = spark.table("workspace.nba_shot_iq.gold_temporadas")

resultados = []

def verificar(nombre, condicion):
    estado = "PASS" if condicion else "FAIL"
    resultados.append((nombre, estado))
    print(f"{estado}: {nombre}")

# Volumen y estructura
verificar("Bronze contiene 703805 registros", bronze.count() == 703805)
verificar("Bronze conserva 19 columnas", len(bronze.columns) == 19)
verificar("Silver contiene 703781 registros", silver.count() == 703781)
verificar("Se eliminaron 24 duplicados", bronze.count() - silver.count() == 24)

# Calidad de los datos
verificar(
    "Sin duplicados exactos en Silver",
    silver.count() == silver.dropDuplicates(bronze.columns).count()
)

verificar(
    "Sin distancias negativas",
    silver.filter(F.col("shot_distance") < 0).count() == 0
)

verificar(
    "Indicador is_made valido",
    silver.filter(
        F.col("is_made").isNull() |
        (~F.col("is_made").isin(0, 1))
    ).count() == 0
)

verificar(
    "Temporadas correctas",
    silver.filter(~F.col("season").isin(2024, 2025, 2026)).count() == 0
)

# Consistencia entre Silver y Gold
verificar(
    "Gold conserva todos los intentos",
    gold.agg(F.sum("intentos")).first()[0] == silver.count()
)

verificar(
    "Gold contiene tres temporadas",
    gold.count() == 3
)

# Resumen
aprobadas = sum(1 for _, estado in resultados if estado == "PASS")
print(f"\nVALIDACIONES APROBADAS: {aprobadas}/{len(resultados)}")

PASS: Bronze contiene 703805 registros
PASS: Bronze conserva 19 columnas
PASS: Silver contiene 703781 registros
PASS: Se eliminaron 24 duplicados
PASS: Sin duplicados exactos en Silver
PASS: Sin distancias negativas
PASS: Indicador is_made valido
PASS: Temporadas correctas
PASS: Gold conserva todos los intentos
PASS: Gold contiene tres temporadas

VALIDACIONES APROBADAS: 10/10


In [0]:
from pyspark.sql import functions as F

# ============================================
# NBA SHOT IQ - DIAGNOSTICO DE DATOS CLUTCH
# ============================================

# Usamos el DataFrame Silver del pipeline
df = silver

# Variables necesarias para investigar clutch
columnas = [
    "game_id",
    "period",
    "clock",
    "score_home",
    "score_away",
    "shot_result",
    "shot_value"
]

# 1. Verificar estructura y tipos
print("1. COLUMNAS Y TIPOS")
df.select(*columnas).printSchema()

# 2. Ver ejemplos reales
print("\n2. EJEMPLOS REALES DE LANZAMIENTOS")
display(df.select(*columnas).limit(15))

# 3. Medir disponibilidad de marcadores
print("\n3. DISPONIBILIDAD DE MARCADORES")

marcador_valido = (
    F.col("score_home").isNotNull()
    & (F.trim(F.col("score_home").cast("string")) != "")
    & F.col("score_away").isNotNull()
    & (F.trim(F.col("score_away").cast("string")) != "")
)

df.select(
    F.count("*").alias("total_tiros"),
    F.sum(
        F.when(marcador_valido, 1).otherwise(0)
    ).alias("tiros_con_ambos_marcadores"),
    F.round(
        F.avg(F.when(marcador_valido, 1).otherwise(0)) * 100,
        2
    ).alias("porcentaje_con_marcador")
).show(truncate=False)

# 4. Revisar formato del reloj
print("\n4. FORMATOS DEL RELOJ")

display(
    df.groupBy("clock")
      .count()
      .orderBy(F.desc("count"))
      .limit(15)
)

# 5. Revisar periodos
print("\n5. DISTRIBUCION POR PERIODO")

df.groupBy("period") \
  .count() \
  .orderBy("period") \
  .show(truncate=False)

print("\nDIAGNOSTICO COMPLETADO")

1. COLUMNAS Y TIPOS
root
 |-- game_id: string (nullable = true)
 |-- period: long (nullable = true)
 |-- clock: string (nullable = true)
 |-- score_home: string (nullable = true)
 |-- score_away: string (nullable = true)
 |-- shot_result: string (nullable = true)
 |-- shot_value: long (nullable = true)


2. EJEMPLOS REALES DE LANZAMIENTOS


game_id,period,clock,score_home,score_away,shot_result,shot_value
0022300001,1,PT10M03.00S,null,null,Missed,2
0022300001,1,PT09M20.00S,null,null,Missed,3
0022300001,1,PT09M11.00S,12,5,Made,2
0022300001,2,PT08M21.00S,45,32,Made,2
0022300001,2,PT00M12.60S,69,51,Made,2
0022300001,3,PT07M31.00S,74,65,Made,2
0022300001,3,PT02M57.00S,84,81,Made,2
0022300001,3,PT01M13.00S,null,null,Missed,3
0022300001,4,PT10M14.00S,null,null,Missed,3
0022300002,1,PT02M30.00S,null,null,Missed,3



3. DISPONIBILIDAD DE MARCADORES
+-----------+--------------------------+-----------------------+
|total_tiros|tiros_con_ambos_marcadores|porcentaje_con_marcador|
+-----------+--------------------------+-----------------------+
|703781     |330699                    |46.99                  |
+-----------+--------------------------+-----------------------+


4. FORMATOS DEL RELOJ


clock,count
PT00M00.00S,2696
PT11M43.00S,1338
PT11M45.00S,1246
PT01M00.00S,1235
PT11M44.00S,1226
PT11M46.00S,1123
PT11M42.00S,1118
PT11M41.00S,1117
PT08M57.00S,1087
PT11M40.00S,1081



5. DISTRIBUCION POR PERIODO
+------+------+
|period|count |
+------+------+
|1     |181930|
|2     |176265|
|3     |174915|
|4     |167009|
|5     |3305  |
|6     |357   |
+------+------+


DIAGNOSTICO COMPLETADO


In [0]:
from pyspark.sql import functions as F

# Mostrar las tablas reales que tenemos guardadas
spark.sql("SHOW TABLES IN workspace.nba_shot_iq").show(
    truncate=False
)

+-----------+---------------+-----------+
|database   |tableName      |isTemporary|
+-----------+---------------+-----------+
|nba_shot_iq|bronze_shots   |false      |
|nba_shot_iq|gold_jugadores |false      |
|nba_shot_iq|gold_temporadas|false      |
|nba_shot_iq|gold_zonas     |false      |
|nba_shot_iq|silver_shots   |false      |
+-----------+---------------+-----------+



In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.nba_shot_iq.silver_shots")

resultado = (
    df.groupBy("shot_result")
    .agg(
        F.count("*").alias("total_tiros"),
        F.sum(
            F.when(
                F.col("score_home").isNotNull() &
                F.col("score_away").isNotNull(),
                1
            ).otherwise(0)
        ).alias("con_marcador")
    )
    .withColumn(
        "porcentaje_con_marcador",
        F.round(
            F.col("con_marcador") /
            F.col("total_tiros") * 100,
            2
        )
    )
)

display(resultado)

shot_result,total_tiros,con_marcador,porcentaje_con_marcador
Missed,373082,0,0.0
Made,330699,330699,100.0


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.nba_shot_iq.silver_shots")

display(
    df.filter(F.col("game_id") == "0022300001")
      .select(
          "game_id",
          "period",
          "clock",
          "player_name",
          "shot_result",
          "shot_value",
          "score_home",
          "score_away"
      )
      .orderBy("period", F.desc("clock"))
      .limit(30)
)

game_id,period,clock,player_name,shot_result,shot_value,score_home,score_away
0022300001,1,PT11M41.00S,Turner,Made,2,2,0
0022300001,1,PT11M23.00S,Mobley,Missed,2,null,null
0022300001,1,PT11M15.00S,Brown,Missed,2,null,null
0022300001,1,PT11M09.00S,Mathurin,Made,3,5,0
0022300001,1,PT10M56.00S,Mitchell,Made,2,5,2
0022300001,1,PT10M25.00S,Mitchell,Made,3,5,5
0022300001,1,PT10M14.00S,Turner,Made,3,8,5
0022300001,1,PT10M03.00S,Mobley,Missed,2,null,null
0022300001,1,PT09M49.00S,Mathurin,Missed,2,null,null
0022300001,1,PT09M36.00S,Mathurin,Missed,2,null,null


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.nba_shot_iq.silver_shots")

# Convertir el reloj PT11M41.00S a segundos
df = df.withColumn(
    "segundos_restantes",
    F.regexp_extract("clock", r"PT(\d+)M", 1).cast("int") * 60
    + F.regexp_extract("clock", r"(\d+(?:\.\d+)?)S", 1).cast("double")
)

# Orden cronológico dentro del partido
ventana = (
    Window.partitionBy("game_id")
    .orderBy(
        F.col("period").asc(),
        F.col("segundos_restantes").desc()
    )
    .rowsBetween(Window.unboundedPreceding, -1)
)

# Último marcador registrado ANTES de cada lanzamiento
df = (
    df.withColumn(
        "marcador_local_previo",
        F.last(
            F.col("score_home").cast("int"),
            ignorenulls=True
        ).over(ventana)
    )
    .withColumn(
        "marcador_visitante_previo",
        F.last(
            F.col("score_away").cast("int"),
            ignorenulls=True
        ).over(ventana)
    )
)

display(
    df.filter(F.col("game_id") == "0022300001")
      .select(
          "period",
          "clock",
          "player_name",
          "shot_result",
          "marcador_local_previo",
          "marcador_visitante_previo"
      )
      .orderBy(
          F.col("period").asc(),
          F.col("segundos_restantes").desc()
      )
      .limit(25)
)

period,clock,player_name,shot_result,marcador_local_previo,marcador_visitante_previo
1,PT11M41.00S,Turner,Made,null,null
1,PT11M23.00S,Mobley,Missed,2,0
1,PT11M15.00S,Brown,Missed,2,0
1,PT11M09.00S,Mathurin,Made,2,0
1,PT10M56.00S,Mitchell,Made,5,0
1,PT10M25.00S,Mitchell,Made,5,2
1,PT10M14.00S,Turner,Made,5,5
1,PT10M03.00S,Mobley,Missed,8,5
1,PT09M49.00S,Mathurin,Missed,8,5
1,PT09M36.00S,Mathurin,Missed,8,5


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.nba_shot_iq.silver_shots")

# Convertir reloj a segundos
df = df.withColumn(
    "segundos_restantes",
    F.regexp_extract("clock", r"PT(\d+)M", 1).cast("int") * 60
    + F.regexp_extract("clock", r"(\d+(?:\.\d+)?)S", 1).cast("double")
)

# Solo tiros con ambos marcadores disponibles
marcadores = (
    df.filter(
        F.col("score_home").isNotNull()
        & F.col("score_away").isNotNull()
    )
    .withColumn("local", F.col("score_home").cast("int"))
    .withColumn("visitante", F.col("score_away").cast("int"))
)

ventana = Window.partitionBy("game_id").orderBy(
    F.col("period").asc(),
    F.col("segundos_restantes").desc()
)

marcadores = (
    marcadores
    .withColumn("local_anterior", F.lag("local").over(ventana))
    .withColumn("visitante_anterior", F.lag("visitante").over(ventana))
    .withColumn("cambio_local", F.col("local") - F.col("local_anterior"))
    .withColumn(
        "cambio_visitante",
        F.col("visitante") - F.col("visitante_anterior")
    )
)

resumen = marcadores.filter(
    F.col("local_anterior").isNotNull()
).agg(
    F.count("*").alias("transiciones_analizadas"),
    F.sum(
        F.when(
            (F.col("cambio_local") < 0)
            | (F.col("cambio_visitante") < 0),
            1
        ).otherwise(0)
    ).alias("marcadores_que_retroceden"),
    F.sum(
        F.when(
            (F.col("cambio_local") + F.col("cambio_visitante")) > 3,
            1
        ).otherwise(0)
    ).alias("cambios_superiores_a_3_puntos")
)

display(resumen)

transiciones_analizadas,marcadores_que_retroceden,cambios_superiores_a_3_puntos
326737,5,50459


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.nba_shot_iq.silver_shots")

resultado = (
    df.filter(
        (F.col("shot_result") == "Made") &
        F.col("score_home").isNotNull() &
        F.col("score_away").isNotNull()
    )
    .groupBy("shot_value")
    .agg(
        F.count("*").alias("tiros_anotados"),
        F.min("score_home").alias("min_local"),
        F.max("score_home").alias("max_local"),
        F.min("score_away").alias("min_visitante"),
        F.max("score_away").alias("max_visitante")
    )
    .orderBy("shot_value")
)

display(resultado)

shot_value,tiros_anotados,min_local,max_local,min_visitante,max_visitante
2,226539,0,99,0,99
3,104160,0,99,0,99


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.nba_shot_iq.silver_shots")

ejemplo = (
    df.filter(F.col("game_id") == "0022300001")
    .withColumn(
        "segundos_restantes",
        F.regexp_extract("clock", r"PT(\d+)M", 1).cast("int") * 60
        + F.regexp_extract("clock", r"(\d+(?:\.\d+)?)S", 1).cast("double")
    )
    .orderBy(
        F.col("period").asc(),
        F.col("segundos_restantes").desc()
    )
    .select(
        "period",
        "clock",
        "team_tricode",
        "player_name",
        "shot_result",
        "shot_value",
        "score_home",
        "score_away"
    )
    .limit(25)
)

display(ejemplo)

period,clock,team_tricode,player_name,shot_result,shot_value,score_home,score_away
1,PT11M41.00S,IND,Turner,Made,2,2,0
1,PT11M23.00S,CLE,Mobley,Missed,2,null,null
1,PT11M15.00S,IND,Brown,Missed,2,null,null
1,PT11M09.00S,IND,Mathurin,Made,3,5,0
1,PT10M56.00S,CLE,Mitchell,Made,2,5,2
1,PT10M25.00S,CLE,Mitchell,Made,3,5,5
1,PT10M14.00S,IND,Turner,Made,3,8,5
1,PT10M03.00S,CLE,Mobley,Missed,2,null,null
1,PT09M49.00S,IND,Mathurin,Missed,2,null,null
1,PT09M36.00S,IND,Mathurin,Missed,2,null,null


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.nba_shot_iq.silver_shots")

df_finales = (
    df.withColumn(
        "segundos_restantes",
        F.regexp_extract("clock", r"PT(\d+)M", 1).cast("int") * 60
        + F.regexp_extract("clock", r"(\d+(?:\.\d+)?)S", 1).cast("double")
    )
    .filter(
        ((F.col("period") == 4) & (F.col("segundos_restantes") <= 300))
        | (F.col("period") >= 5)
    )
)

resumen = df_finales.agg(
    F.count("*").alias("tiros_en_ventana_final"),
    F.sum(
        F.when(
            F.col("score_home").isNotNull()
            & F.col("score_away").isNotNull(),
            1
        ).otherwise(0)
    ).alias("tiros_con_marcador"),
    F.sum(
        F.when(F.col("shot_result") == "Made", 1).otherwise(0)
    ).alias("tiros_acertados"),
    F.countDistinct("game_id").alias("partidos")
)

display(resumen)

tiros_en_ventana_final,tiros_con_marcador,tiros_acertados,partidos
74122,33841,33841,3962


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.nba_shot_iq.silver_shots")

# Convertir el reloj a segundos restantes
df = df.withColumn(
    "segundos_restantes",
    F.regexp_extract("clock", r"PT(\d+)M", 1).cast("int") * 60
    + F.regexp_extract("clock", r"(\d+(?:\.\d+)?)S", 1).cast("double")
)

# Tiempo transcurrido desde el inicio del partido
df = df.withColumn(
    "tiempo_transcurrido",
    F.when(
        F.col("period") <= 4,
        (F.col("period") - 1) * 720
        + 720 - F.col("segundos_restantes")
    ).otherwise(
        2880 + (F.col("period") - 5) * 300
        + 300 - F.col("segundos_restantes")
    )
)

# Marcadores conocidos, registrados después de un enceste
conocidos = (
    df.filter(
        F.col("score_home").isNotNull()
        & F.col("score_away").isNotNull()
    )
    .select(
        "game_id",
        "tiempo_transcurrido",
        F.col("score_home").cast("int").alias("local_conocido"),
        F.col("score_away").cast("int").alias("visitante_conocido")
    )
)

# Agrupar por instante para evitar inventar un orden entre tiros simultáneos
instantes = (
    df.select("game_id", "tiempo_transcurrido")
    .distinct()
)

marcadores_por_instante = (
    conocidos.groupBy("game_id", "tiempo_transcurrido")
    .agg(
        F.max("local_conocido").alias("local_en_instante"),
        F.max("visitante_conocido").alias("visitante_en_instante")
    )
)

linea = instantes.join(
    marcadores_por_instante,
    ["game_id", "tiempo_transcurrido"],
    "left"
)

ventana = (
    Window.partitionBy("game_id")
    .orderBy("tiempo_transcurrido")
    .rowsBetween(Window.unboundedPreceding, -1)
)

linea = (
    linea
    .withColumn(
        "marcador_local_estimado",
        F.last("local_en_instante", ignorenulls=True).over(ventana)
    )
    .withColumn(
        "marcador_visitante_estimado",
        F.last("visitante_en_instante", ignorenulls=True).over(ventana)
    )
)

df_clutch = df.join(
    linea.select(
        "game_id",
        "tiempo_transcurrido",
        "marcador_local_estimado",
        "marcador_visitante_estimado"
    ),
    ["game_id", "tiempo_transcurrido"],
    "left"
)

df_clutch = (
    df_clutch
    .withColumn(
        "diferencia_estimada",
        F.abs(
            F.col("marcador_local_estimado")
            - F.col("marcador_visitante_estimado")
        )
    )
    .withColumn(
        "ventana_final",
        (
            (F.col("period") == 4)
            & (F.col("segundos_restantes") <= 300)
        ) | (F.col("period") >= 5)
    )
    .withColumn(
        "candidato_clutch",
        F.col("ventana_final")
        & (F.col("diferencia_estimada") <= 5)
    )
)

resumen = df_clutch.agg(
    F.sum(F.col("ventana_final").cast("int")).alias("tiros_finales"),
    F.sum(
        F.col("candidato_clutch").cast("int")
    ).alias("candidatos_clutch")
)

display(resumen)

tiros_finales,candidatos_clutch
74122,23636


In [0]:
from pyspark.sql import functions as F

jugadores = (
    df_clutch
    .filter(F.col("candidato_clutch") == True)
    .groupBy("player_name")
    .agg(
        F.count("*").alias("tiros_clutch"),
        F.sum(
            F.when(F.col("shot_result") == "Made", 1)
             .otherwise(0)
        ).alias("encestados")
    )
    .filter(F.col("tiros_clutch") >= 50)
    .withColumn(
        "efectividad_clutch",
        F.round(
            F.col("encestados") / F.col("tiros_clutch") * 100,
            2
        )
    )
    .orderBy(F.desc("efectividad_clutch"))
)

display(jugadores.limit(20))

player_name,tiros_clutch,encestados,efectividad_clutch
Sabonis,74,45,60.81
Duren,65,38,58.46
Zubac,60,35,58.33
Mobley,80,46,57.5
Edgecombe,51,29,56.86
Gordon,113,64,56.64
Okongwu,52,29,55.77
Hachimura,52,29,55.77
Jones Jr.,54,30,55.56
Rollins,56,31,55.36


In [0]:
from pyspark.sql import functions as F

# 1. Efectividad general por jugador
general = (
    df
    .groupBy("person_id")
    .agg(
        F.first("player_name", ignorenulls=True).alias("jugador"),
        F.count("*").alias("tiros_totales"),
        F.sum(
            F.when(F.col("shot_result") == "Made", 1)
             .otherwise(0)
        ).alias("encestados_totales")
    )
)

# 2. Efectividad de candidatos clutch
clutch = (
    df_clutch
    .filter(F.col("candidato_clutch") == True)
    .groupBy("person_id")
    .agg(
        F.count("*").alias("tiros_clutch"),
        F.sum(
            F.when(F.col("shot_result") == "Made", 1)
             .otherwise(0)
        ).alias("encestados_clutch")
    )
)

# 3. Comparación
comparacion = (
    general
    .join(clutch, on="person_id", how="inner")
    .filter(
        (F.col("tiros_clutch") >= 50) &
        (F.col("tiros_totales") >= 200)
    )
    .withColumn(
        "efectividad_general",
        F.round(
            F.col("encestados_totales") /
            F.col("tiros_totales") * 100, 2
        )
    )
    .withColumn(
        "efectividad_clutch",
        F.round(
            F.col("encestados_clutch") /
            F.col("tiros_clutch") * 100, 2
        )
    )
    .withColumn(
        "diferencia_pp",
        F.round(
            F.col("efectividad_clutch") -
            F.col("efectividad_general"), 2
        )
    )
    .select(
        "jugador",
        "tiros_totales",
        "tiros_clutch",
        "efectividad_general",
        "efectividad_clutch",
        "diferencia_pp"
    )
    .orderBy(F.desc("diferencia_pp"))
)

display(comparacion.limit(20))

jugador,tiros_totales,tiros_clutch,efectividad_general,efectividad_clutch,diferencia_pp
Edgecombe,1191,51,43.49,56.86,13.37
Grant,2209,101,42.92,51.49,8.57
Rollins,1341,56,47.43,55.36,7.93
Allen,2222,50,65.89,72.0,6.11
Murray,4095,204,46.86,52.94,6.08
Gordon,1926,88,53.06,59.09,6.03
Jones Jr.,1680,54,50.06,55.56,5.5
Hield,2133,69,42.76,47.83,5.07
Monk,2464,128,44.03,48.44,4.41
Sharpe,2447,91,44.34,48.35,4.01


In [0]:
from pyspark.sql import functions as F

df_clutch.groupBy("candidato_clutch").agg(
    F.count("*").alias("total_tiros"),
    F.sum(
        F.when(F.col("shot_result") == "Made", 1).otherwise(0)
    ).alias("encestados"),
    F.countDistinct("game_id").alias("partidos")
).withColumn(
    "efectividad",
    F.round(F.col("encestados") / F.col("total_tiros") * 100, 2)
).show(truncate=False)

+----------------+-----------+----------+--------+-----------+
|candidato_clutch|total_tiros|encestados|partidos|efectividad|
+----------------+-----------+----------+--------+-----------+
|false           |680145     |320411    |3962    |47.11      |
|true            |23636      |10288     |1760    |43.53      |
+----------------+-----------+----------+--------+-----------+



In [0]:
from pyspark.sql import functions as F

df_clutch.filter(
    F.col("candidato_clutch") == True
).groupBy("period").agg(
    F.count("*").alias("total_tiros"),
    F.min("clock").alias("ejemplo_reloj_min"),
    F.max("clock").alias("ejemplo_reloj_max")
).orderBy("period").show(truncate=False)

+------+-----------+-----------------+-----------------+
|period|total_tiros|ejemplo_reloj_min|ejemplo_reloj_max|
+------+-----------+-----------------+-----------------+
|4     |20404      |PT00M00.00S      |PT05M00.00S      |
|5     |2911       |PT00M00.00S      |PT04M57.00S      |
|6     |321        |PT00M00.00S      |PT04M53.00S      |
+------+-----------+-----------------+-----------------+



In [0]:
from pyspark.sql import functions as F

revision = (
    df_clutch
    .filter(F.col("candidato_clutch") == True)
    .withColumn(
        "diferencia_estimada",
        F.abs(
            F.col("marcador_local_estimado") -
            F.col("marcador_visitante_estimado")
        )
    )
)

revision.agg(
    F.count("*").alias("total_candidatos"),

    F.sum(
        F.when(
            F.col("diferencia_estimada") <= 5, 1
        ).otherwise(0)
    ).alias("diferencia_valida"),

    F.sum(
        F.when(
            F.col("diferencia_estimada") > 5, 1
        ).otherwise(0)
    ).alias("diferencia_invalida"),

    F.sum(
        F.when(
            F.col("diferencia_estimada").isNull(), 1
        ).otherwise(0)
    ).alias("marcador_desconocido")

).show(truncate=False)

+----------------+-----------------+-------------------+--------------------+
|total_candidatos|diferencia_valida|diferencia_invalida|marcador_desconocido|
+----------------+-----------------+-------------------+--------------------+
|23636           |23636            |0                  |0                   |
+----------------+-----------------+-------------------+--------------------+



In [0]:
print("COLUMNAS DE LINEA:")
print(linea.columns)

print("\nCOLUMNAS DE CLUTCH:")
print(df_clutch.columns)

print("\nDEFINICION DE VENTANA:")
print(ventana)

COLUMNAS DE LINEA:
['game_id', 'tiempo_transcurrido', 'local_en_instante', 'visitante_en_instante', 'marcador_local_estimado', 'marcador_visitante_estimado']

COLUMNAS DE CLUTCH:
['game_id', 'tiempo_transcurrido', 'season', 'season_type_id', 'period', 'clock', 'team_id', 'team_tricode', 'person_id', 'player_name', 'action_type', 'sub_type', 'shot_result', 'shot_value', 'shot_distance', 'x_legacy', 'y_legacy', 'description', 'score_home', 'score_away', 'is_made', 'shot_points', 'shot_zone', 'segundos_restantes', 'marcador_local_estimado', 'marcador_visitante_estimado', 'diferencia_estimada', 'ventana_final', 'candidato_clutch']

DEFINICION DE VENTANA:
WindowSpec(PartitionBy(game_id), OrderBy(tiempo_transcurrido ASC NULLS FIRST), WindowFrame(ROW_FRAME, -9223372036854775808, -1))


In [0]:
from pyspark.sql import functions as F

empates = (
    df_clutch
    .groupBy("game_id", "tiempo_transcurrido")
    .count()
    .filter(F.col("count") > 1)
)

print("Instantes con varios lanzamientos:", empates.count())
print("Lanzamientos adicionales en instantes repetidos:",
      empates.select(F.sum(F.col("count") - 1)).first()[0] or 0)

display(empates.orderBy(F.desc("count")).limit(10))

Instantes con varios lanzamientos: 1162
Lanzamientos adicionales en instantes repetidos: 1190


game_id,tiempo_transcurrido,count
0022500317,340.0,4
0022301223,320.0,3
0022400109,440.0,3
0022400560,1107.0,3
0022301085,1305.0,3
0022300100,2433.0,3
0042300155,373.0,3
0022500289,264.0,3
0022400444,2529.0,3
0022400662,2329.0,3


In [0]:
from pyspark.sql import functions as F

empates_clutch = (
    df_clutch
    .join(
        empates.select("game_id", "tiempo_transcurrido"),
        ["game_id", "tiempo_transcurrido"],
        "inner"
    )
)

resultado = empates_clutch.agg(
    F.count("*").alias("tiros_en_instantes_empatados"),
    F.sum(
        F.when(F.col("candidato_clutch") == True, 1).otherwise(0)
    ).alias("candidatos_clutch_afectados")
)

display(resultado)

tiros_en_instantes_empatados,candidatos_clutch_afectados
2352,153


In [0]:
from pyspark.sql import functions as F

clutch_conservador = (
    df_clutch
    .join(
        empates.select("game_id", "tiempo_transcurrido")
               .withColumn("tiempo_ambiguo", F.lit(True)),
        ["game_id", "tiempo_transcurrido"],
        "left"
    )
    .filter(F.col("tiempo_ambiguo").isNull())
)

resumen = (
    clutch_conservador
    .filter(F.col("candidato_clutch") == True)
    .agg(
        F.count("*").alias("candidatos_clutch"),
        F.sum(
            F.when(F.col("shot_result") == "Made", 1).otherwise(0)
        ).alias("encestados")
    )
    .withColumn(
        "efectividad",
        F.round(F.col("encestados") / F.col("candidatos_clutch") * 100, 2)
    )
)

display(resumen)

candidatos_clutch,encestados,efectividad
23483,10228,43.55
